# Análise de um banco de dados de livros com SQL

**Autora:** Luciana Jost

**Bootcamp de Análise de Dados - TripleTen**

**Data:** outubro de 2026

## 1. Objetivos do estudo

Com a mudança de rotina provocada pela pandemia, a leitura ganhou espaço no tempo livre das pessoas, e startups passaram a desenvolver aplicativos para leitores. Este estudo analisa o banco de dados de um serviço concorrente nesse mercado, com informações sobre livros, autores, editoras, classificações e avaliações dos usuários.

O objetivo é extrair informações que ajudem a formular a proposta de valor de um novo produto. Para isso, o estudo responde a cinco perguntas:

1. Quantos livros foram lançados depois de 1 de janeiro de 2000?
2. Qual é o número de avaliações e a classificação média de cada livro?
3. Qual editora lançou o maior número de livros com mais de 50 páginas?
4. Qual autor tem a classificação média mais alta, considerando apenas livros com pelo menos 50 classificações?
5. Qual é o número médio de avaliações entre os usuários que classificaram mais de 50 livros?

**Sobre os termos:** neste estudo, "classificação" é a nota atribuída por um usuário a um livro, registrada na tabela `ratings`. "Avaliação" é o texto escrito por um usuário sobre um livro, registrado na tabela `reviews`.

### Estrutura da análise

1. Objetivos do estudo
2. Conexão com o banco de dados e estudo das tabelas
3. Consultas
4. Conclusões

## 2. Conexão com o banco de dados e estudo das tabelas

### 2.1 Conexão

A conexão com o banco de dados segue as instruções fornecidas pelo bootcamp. A função `executar_consulta` recebe uma consulta SQL e devolve o resultado em um DataFrame. O pandas é usado apenas para executar as consultas e exibir os resultados.

**Sobre as credenciais:** por segurança, usuário, senha e endereço do banco não ficam escritos no notebook. Eles são lidos de variáveis de ambiente (`DB_USER`, `DB_PASSWORD`, `DB_HOST`, `DB_PORT` e `DB_NAME`), definidas fora do projeto. O banco pertence ao ambiente do bootcamp e não é público, por isso os resultados exibidos neste notebook são os da execução original.

In [29]:
import os

import pandas as pd
from sqlalchemy import create_engine

# As credenciais são lidas de variáveis de ambiente e não ficam registradas no código
db_config = {
    'user': os.environ['DB_USER'],
    'pwd': os.environ['DB_PASSWORD'],
    'host': os.environ['DB_HOST'],
    'port': int(os.environ.get('DB_PORT', 5432)),
    'db': os.environ['DB_NAME'],
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})


def executar_consulta(query):
    """Executa uma consulta SQL no banco de dados.

    Recebe o texto da consulta e retorna o resultado em um DataFrame.
    """
    return pd.io.sql.read_sql(query, con=engine)

### 2.2 Estudo das tabelas

O banco de dados tem cinco tabelas. A célula a seguir mostra o número de linhas e as primeiras linhas de cada uma.

In [30]:
TABELAS = ['books', 'authors', 'publishers', 'ratings', 'reviews']

for tabela in TABELAS:
    total = executar_consulta(f'SELECT COUNT(*) AS total FROM {tabela}')
    print(f'Tabela: {tabela} ({total.loc[0, "total"]} linhas)')
    display(executar_consulta(f'SELECT * FROM {tabela} LIMIT 5'))

Tabela: books (1000 linhas)


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


Tabela: authors (636 linhas)


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


Tabela: publishers (340 linhas)


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


Tabela: ratings (6456 linhas)


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


Tabela: reviews (2793 linhas)


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


**Leitura das tabelas:**

- `books` tem 1.000 livros, com título, número de páginas, data de publicação e os identificadores do autor e da editora.
- `authors` tem 636 autores. Em alguns registros, o campo reúne mais de um nome, separados por barra, como em obras com coautores ou ilustradores.
- `publishers` tem 340 editoras. Há nomes muito parecidos entre si, como "Ace", "Ace Book" e "Ace Books", registrados como editoras diferentes.
- `ratings` tem 6.456 classificações, cada uma com o livro, o usuário e a nota atribuída.
- `reviews` tem 2.793 avaliações, cada uma com o livro, o usuário e o texto escrito.

As tabelas se relacionam pelos identificadores: `book_id` liga `books` a `ratings` e a `reviews`, `author_id` liga `books` a `authors` e `publisher_id` liga `books` a `publishers`.

O número de classificações (6.456) é maior do que o de avaliações (2.793): nem todo usuário que dá uma nota escreve um texto.

## 3. Consultas

### 3.1 Livros lançados depois de 1 de janeiro de 2000

A consulta conta os livros cuja data de publicação é posterior a 1 de janeiro de 2000.

In [31]:
query = '''
SELECT
    COUNT(*) AS livros_lancados_depois_de_2000
FROM books
WHERE publication_date > '2000-01-01';
'''

executar_consulta(query)

,livros_lancados_depois_de_2000
0,819


**Resultado:** 819 livros foram lançados depois de 1 de janeiro de 2000.

**Conclusão:** dos 1.000 livros do catálogo, 819 (81,9%) foram publicados depois de 1 de janeiro de 2000, e os publicados até essa data representam menos de um quinto do total. O acervo do serviço concorrente é formado principalmente por edições recentes. Isso sugere que títulos das últimas duas décadas devem ter peso importante em um catálogo competitivo. Os dados, porém, mostram o que o concorrente oferece, e não o que os leitores procuram: confirmar essa prioridade exigiria dados de uso ou de vendas.

### 3.2 Número de avaliações e classificação média de cada livro

A consulta calcula, para cada livro, o número de avaliações em texto (tabela `reviews`) e a média das notas recebidas (tabela `ratings`). As duas contas são feitas em subconsultas separadas e depois unidas à tabela de livros, para que todos os livros apareçam no resultado, inclusive os que não têm avaliações.

In [32]:
query = '''
SELECT
    b.book_id,
    b.title,
    COALESCE(rv.numero_de_avaliacoes, 0) AS numero_de_avaliacoes,
    ROUND(rt.classificacao_media, 2) AS classificacao_media
FROM books AS b
LEFT JOIN (
    SELECT
        book_id,
        COUNT(review_id) AS numero_de_avaliacoes
    FROM reviews
    GROUP BY book_id
) AS rv ON rv.book_id = b.book_id
LEFT JOIN (
    SELECT
        book_id,
        AVG(rating) AS classificacao_media
    FROM ratings
    GROUP BY book_id
) AS rt ON rt.book_id = b.book_id
ORDER BY numero_de_avaliacoes DESC, classificacao_media DESC;
'''

avaliacoes_por_livro = executar_consulta(query)
avaliacoes_por_livro

,book_id,title,numero_de_avaliacoes,classificacao_media
0,948,Twilight (Twilight #1),7,3.66
1,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.41
2,299,Harry Potter and the Chamber of Secrets (Harry...,6,4.29
3,656,The Book Thief,6,4.26
4,734,The Glass Castle,6,4.21
...,...,...,...,...
995,387,Leonardo's Notebooks,0,4.00
996,191,Disney's Beauty and the Beast (A Little Golden...,0,4.00
997,221,Essential Tales and Poems,0,4.00
998,83,Anne Rice's The Vampire Lestat: A Graphic Novel,0,3.67


Como o resultado tem uma linha por livro, a consulta a seguir resume a tabela: total de livros, livros sem avaliação, maior número de avaliações, média de avaliações por livro e média das classificações.

In [33]:
query = '''
WITH por_livro AS (
    SELECT
        b.book_id,
        COALESCE(rv.numero_de_avaliacoes, 0) AS numero_de_avaliacoes,
        rt.classificacao_media
    FROM books AS b
    LEFT JOIN (
        SELECT
            book_id,
            COUNT(review_id) AS numero_de_avaliacoes
        FROM reviews
        GROUP BY book_id
    ) AS rv ON rv.book_id = b.book_id
    LEFT JOIN (
        SELECT
            book_id,
            AVG(rating) AS classificacao_media
        FROM ratings
        GROUP BY book_id
    ) AS rt ON rt.book_id = b.book_id
)
SELECT
    COUNT(*) AS total_de_livros,
    SUM(CASE WHEN numero_de_avaliacoes = 0 THEN 1 ELSE 0 END) AS livros_sem_avaliacao,
    MAX(numero_de_avaliacoes) AS maximo_de_avaliacoes,
    ROUND(AVG(numero_de_avaliacoes), 2) AS media_de_avaliacoes_por_livro,
    ROUND(AVG(classificacao_media), 2) AS media_das_classificacoes
FROM por_livro;
'''

executar_consulta(query)

,total_de_livros,livros_sem_avaliacao,maximo_de_avaliacoes,media_de_avaliacoes_por_livro,media_das_classificacoes
0,1000,6,7,2.79,3.9


**Resultado:** a consulta retorna uma linha para cada um dos 1.000 livros, com o número de avaliações e a classificação média.

- O número de avaliações por livro varia de 0 a 7, com média de 2,79.
- O livro mais avaliado é *Twilight (Twilight #1)*, com 7 avaliações e classificação média de 3,66.
- Seis livros não têm nenhuma avaliação em texto.
- A média das classificações médias dos livros é 3,90, em uma escala que vai até 5. A coluna `media_das_classificacoes` do resumo é, portanto, uma média das médias por livro, em que cada livro tem o mesmo peso.

**Conclusão:** os usuários escrevem poucas avaliações: em média, menos de três por livro. Os títulos mais avaliados são obras populares e de grande público, como *Twilight* e a série *Harry Potter*. O livro com mais avaliações, porém, não é o mais bem classificado: *Twilight* tem média de 3,66, abaixo da média geral. Volume de avaliações e qualidade percebida são medidas diferentes, e o novo produto deve acompanhar as duas. As classificações em geral são altas, o que pode sugerir que os usuários tendem a dar nota a livros de que gostaram, hipótese que estes dados não permitem confirmar.

### 3.3 Editora com o maior número de livros com mais de 50 páginas

A consulta considera apenas os livros com mais de 50 páginas, para excluir brochuras e publicações semelhantes, e conta quantos livros cada editora lançou. O resultado mostra as cinco editoras com mais livros, para dar contexto à primeira colocada.

In [34]:
query = '''
SELECT
    p.publisher AS editora,
    COUNT(b.book_id) AS numero_de_livros
FROM books AS b
INNER JOIN publishers AS p ON p.publisher_id = b.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher_id, p.publisher
ORDER BY numero_de_livros DESC
LIMIT 5;
'''

executar_consulta(query)

,editora,numero_de_livros
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Bantam,19


**Resultado:** a editora que lançou o maior número de livros com mais de 50 páginas é a **Penguin Books**, com 42 livros.

Em seguida vêm a Vintage (31 livros), a Grand Central Publishing (25), a Penguin Classics (24) e a Bantam (19).

**Conclusão:** a Penguin Books lidera com folga: tem 11 livros a mais do que a segunda colocada. A quarta posição é ocupada pela Penguin Classics, registrada no banco como uma editora separada. Somadas, as duas marcas Penguin respondem por 66 livros, o que reforça o peso desse grupo editorial no catálogo. Como observado no estudo das tabelas, o banco registra nomes parecidos como editoras distintas, de modo que a concentração real por grupo editorial pode ser maior do que a contagem por nome indica. Para o novo produto, as editoras do topo da lista são as parcerias de catálogo mais relevantes.

### 3.4 Autor com a classificação média mais alta

A consulta considera apenas os livros com pelo menos 50 classificações. Para cada um deles, calcula a classificação média. Em seguida, calcula a média dessas classificações por autor. O resultado mostra os cinco autores mais bem classificados e o número de livros de cada um que entrou na conta.

In [35]:
query = '''
SELECT
    a.author AS autor,
    COUNT(lv.book_id) AS livros_considerados,
    ROUND(AVG(lv.classificacao_media), 2) AS classificacao_media
FROM (
    SELECT
        book_id,
        AVG(rating) AS classificacao_media
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
) AS lv
INNER JOIN books AS b ON b.book_id = lv.book_id
INNER JOIN authors AS a ON a.author_id = b.author_id
GROUP BY a.author_id, a.author
ORDER BY AVG(lv.classificacao_media) DESC
LIMIT 5;
'''

executar_consulta(query)

,autor,livros_considerados,classificacao_media
0,J.K. Rowling/Mary GrandPré,4,4.28
1,Markus Zusak/Cao Xuân Việt Khương,1,4.26
2,J.R.R. Tolkien,2,4.26
3,Louisa May Alcott,1,4.19
4,Rick Riordan,1,4.08


**Resultado:** o autor com a classificação média mais alta, entre os livros com pelo menos 50 classificações, é **J.K. Rowling/Mary GrandPré**, com média de 4,28 em quatro livros.

Em seguida vêm Markus Zusak/Cao Xuân Việt Khương (4,26, um livro), J.R.R. Tolkien (4,26, dois livros), Louisa May Alcott (4,19, um livro) e Rick Riordan (4,08, um livro).

A classificação de cada autor foi calculada como a média das classificações médias dos seus livros, de modo que cada livro tem o mesmo peso no resultado.

**Conclusão:** J.K. Rowling, registrada junto com a ilustradora Mary GrandPré, lidera com a média mais alta e com o maior número de livros considerados. O resultado dela é o mais sólido da lista: a média entre os seus quatro livros é de 4,28, enquanto três dos cinco autores entram com um único livro. As diferenças no topo são pequenas, de dois centésimos entre a primeira e a segunda posição.

Vale lembrar que o filtro é por livro, e não por usuário: a lista mostra os autores mais bem classificados entre os livros com pelo menos 50 classificações. Como a base não tem uma coluna de gênero, qualquer leitura por gênero vem de conhecimento externo e deve ser tratada com cautela. Parte da lista é de fantasia e literatura juvenil (J.K. Rowling, J.R.R. Tolkien e Rick Riordan), mas não toda: Louisa May Alcott e Markus Zusak escreveram romances que não são de fantasia.

### 3.5 Número médio de avaliações entre os usuários que classificaram mais de 50 livros

A consulta identifica os usuários que classificaram mais de 50 livros, conta quantas avaliações em texto cada um escreveu e calcula a média entre eles. O resultado mostra também quantos usuários atendem ao critério.

In [36]:
query = '''
SELECT
    COUNT(*) AS usuarios_com_mais_de_50_classificacoes,
    ROUND(AVG(numero_de_avaliacoes), 2) AS media_de_avaliacoes
FROM (
    SELECT
        u.username,
        COUNT(rv.review_id) AS numero_de_avaliacoes
    FROM (
        SELECT
            username
        FROM ratings
        GROUP BY username
        HAVING COUNT(DISTINCT book_id) > 50
    ) AS u
    LEFT JOIN reviews AS rv ON rv.username = u.username
    GROUP BY u.username
) AS por_usuario;
'''

executar_consulta(query)

,usuarios_com_mais_de_50_classificacoes,media_de_avaliacoes
0,6,24.33


**Resultado:** seis usuários classificaram mais de 50 livros. Entre eles, o número médio de avaliações em texto é de **24,33**.

**Conclusão:** os usuários mais ativos do serviço são poucos: apenas seis ultrapassam a marca de 50 livros classificados. Esses usuários escrevem, em média, cerca de 24 avaliações cada um. Como todos classificaram mais de 50 livros, isso indica, de forma aproximada, que escrevem um texto para menos da metade dos livros a que dão nota. A consulta não calcula essa proporção usuário a usuário, por isso a leitura deve ser tomada como uma aproximação.

Individualmente, esses usuários são os que mais escrevem. Como grupo, porém, os seis somam cerca de 146 das 2.793 avaliações da base, cerca de 5%. Isso sugere que a maior oportunidade para o novo produto está em levar os muitos usuários menos ativos a escrever avaliações, e não apenas em estimular quem já escreve.

## 4. Conclusões

Este estudo analisou o banco de dados de um serviço de leitura concorrente, com 1.000 livros, 636 autores, 340 editoras, 6.456 classificações e 2.793 avaliações. Todos os resultados foram obtidos com consultas SQL.

### Resultados

| Pergunta | Resultado |
|---|---|
| Livros lançados depois de 1 de janeiro de 2000 | 819 livros, 81,9% do catálogo |
| Avaliações e classificação média por livro | De 0 a 7 avaliações por livro, com média de 2,79. A média das classificações é 3,90 |
| Editora com mais livros com mais de 50 páginas | Penguin Books, com 42 livros |
| Autor com a classificação média mais alta | J.K. Rowling/Mary GrandPré, com média de 4,28 em quatro livros |
| Média de avaliações entre usuários com mais de 50 livros classificados | 24,33 avaliações, entre seis usuários |

### O que os resultados sugerem para o novo produto

Os resultados descrevem o catálogo e o comportamento dos usuários de um concorrente. As recomendações abaixo são hipóteses de trabalho, e confirmá-las exigiria análises adicionais, com dados de demanda, de uso ou de vendas.

- **Catálogo:** o acervo do concorrente é formado em sua maior parte por edições publicadas depois de 2000. Isso sugere que títulos recentes devem ter peso importante em um catálogo competitivo.
- **Parcerias editoriais:** a Penguin Books lidera em número de títulos, e a Penguin Classics aparece entre as cinco primeiras. Isso sugere que as editoras do topo da lista são candidatas naturais a parcerias de catálogo.
- **Autores:** entre os livros com pelo menos 50 classificações, os autores mais bem classificados são de ficção de grande público. A base não tem informação de gênero, então uma leitura por gênero literário exigiria dados adicionais.
- **Popularidade e aprovação:** o livro com mais avaliações, *Twilight*, tem classificação média abaixo da média geral. Volume de avaliações e nota são medidas diferentes e devem ser acompanhadas separadamente.
- **Engajamento:** os usuários dão notas com muito mais frequência do que escrevem avaliações. Há 6.456 classificações para 2.793 avaliações, e, de forma aproximada, mesmo os usuários mais ativos escrevem sobre menos da metade dos livros que classificam. Como os seis usuários mais ativos respondem por apenas cerca de 5% das avaliações, a oportunidade sugerida está em incentivar a escrita entre os usuários menos ativos.

### Limitações

- O banco registra nomes parecidos como editoras diferentes, o que pode subestimar a concentração por grupo editorial.
- Em alguns registros, o campo de autor reúne mais de um nome, de modo que a classificação é atribuída ao conjunto, e não a cada pessoa.
- Apenas seis usuários classificaram mais de 50 livros. A média calculada para esse grupo se apoia em poucas pessoas.